# Lab 2: Your first LLM calls
**Module 1.2: LLM basics for architects.**  **Time:** 25 minutes.

**Goal:** see the levers an architect controls on every LLM call: the prompt, the system prompt,
temperature, output format and model choice, and what each costs in tokens, latency and money.

Run the setup cell first. It works the same in Google Colab and in VS Code.

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();

## 1. One call
`lk.chat()` sends one request to the model and returns the text. Under the hood it uses the standard
`openai` Python client pointed at Groq (or Gemini). Open `labkit.py` and look at `chat()`: it is about 20 lines.

In [ ]:
answer = lk.chat("In 3 short bullet points, what does an 'AI platform' give an enterprise?", tag="first")
print(answer)

## 2. Tokens, latency and cost
Every call is logged. Tokens are what you pay for. `est_cost_usd` shows what this call would cost at the
provider's paid price (we are on a free tier).

In [ ]:
lk.calls_df()

## 3. The system prompt sets behaviour
Same question, two different system prompts.

In [ ]:
q = "Should we build our own LLM?"
print("WITHOUT system prompt:\n", lk.chat(q, max_tokens=150, tag="no-system"), "\n")
print("WITH system prompt:\n", lk.chat(q, system="You advise a CIO. Answer in 2 sentences, plainly, no hype.",
                                       max_tokens=150, tag="system"))

## 4. Structured output (JSON)
Applications need data, not prose. Ask for JSON and parse it into a Python dict.

In [ ]:
email = """Hi team, our checkout API in the Mumbai region has been timing out since 9am.
About 1,200 customers are affected. Please treat this as urgent. Priya, Payments team"""
data = lk.chat_json("Extract: team, region, affected_customers (number), urgency (low/medium/high), summary.\n\n" + email,
                    tag="json")
data

In [ ]:
print("Affected customers + 10%:", round(int(data.get("affected_customers", 0)) * 1.1))

## 5. Streaming
Chat interfaces show text as it is generated. Same model, same cost, better perceived speed.

In [ ]:
_ = lk.stream("Write a 4-line poem about cloud architecture.")

## 6. Compare two models
The same prompt on the big model and the small model. Look at quality, latency and estimated cost.

In [ ]:
prompt = "Explain retrieval-augmented generation (RAG) to a bank's risk committee in 3 sentences."
for m in ["big", "small"]:
    print(f"--- {lk.model_name(m)} ---")
    print(lk.chat(prompt, model=m, tag=f"compare-{m}"), "\n")

lk.calls_df().query("tag.str.startswith('compare')", engine="python")[["model", "latency_s", "prompt_tokens", "completion_tokens", "est_cost_usd"]]

## Try this 1: temperature (3 minutes)
The next cell asks the same question 3 times at **temperature 1.0** (more random). Run it and compare the three answers.

In [ ]:
for i in range(3):
    print(f"run {i+1}:", lk.chat("Suggest a name for an internal HR assistant. Reply with the name only.",
                                temperature=1.0, max_tokens=20, tag="temperature"))

Now change `temperature=1.0` to `temperature=0` in the cell above and run it again. The three answers should be the
same, or nearly the same.

**Question:** for which tasks in your organisation would you want temperature 0, and for which a higher value?

## Try this 2: missing information (2 minutes)
This email does not mention a region. Run the cell and look at the value the model returns for `region`.

In [ ]:
email2 = "Hi team, checkout is failing for some customers since this morning. Please look into it. Ankit"
lk.chat_json("Extract: team, region, affected_customers (number), urgency (low/medium/high), summary.\n\n" + email2,
             tag="json-missing")

**Question:** did the model leave `region` empty, write `null`, or invent a value? What should application code check
before it uses this data?

## Discussion (end of lab)
Which of the levers you used today (system prompt, temperature, output format, model choice) belongs in application
code, and which should a central platform team control?